# Novidades do MissingPatterns.jl 0.7 — um exemplo em português

O [MissingPatterns.jl](https://github.com/dantebertuzzi/MissingPatterns.jl) responde a uma pergunta que
vem antes de qualquer modelo: *o que está faltando aqui, e por quê?* A versão 0.7 traz três
diagnósticos que vão além de *onde* falta para *por que* falta:

- **`missingsentinels`** — encontra a ausência disfarçada de valor (`9` = ignorado, `""`, `"Ignorado"`);
- **`missingmcar`** — o teste de Little para *ausência completamente aleatória* (MCAR);
- **`missingpredictors`** — de quais variáveis a ausência de uma coluna depende.

E melhora três visualizações: padrões ordenados por estrutura, agrupamento sem repetir a coluna do
grupo, e comparação de duas versões de um arquivo por chave.

Cada seção traz a fórmula do que está sendo calculado. A [página Methods](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/methods/) da
documentação tem todas elas, e a [página References](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/references/) as fontes.

**No Colab, escolha antes o ambiente Julia:** *Ambiente de execução ▸ Alterar o tipo de ambiente ▸ Julia*.

In [ ]:
using Pkg
Pkg.add(["MissingPatterns", "DataFrames"])
Pkg.status("MissingPatterns")   # precisa da 0.7 ou posterior

In [ ]:
using MissingPatterns, DataFrames, Dates, Random

## Os dados

Uma tabela simulada de 1.500 notificações, no formato das bases do SUS. Os buracos foram colocados
de propósito, por três mecanismos que não aparecem na tabela:

- **`renda`** é recusada com mais frequência no **Sudeste** e por quem tem **mais de 65 anos** — falta
  *por causa* de outras variáveis (MAR, *missing at random*);
- **`glicemia`** e **`peso`** vêm da mesma consulta, que deixou de acontecer com frequência entre
  **março de 2020 e dezembro de 2021** — a ausência depende da data, não de nenhum valor medido;
- **`escolaridade`** nunca é `missing`: o "ignorado" vem codificado como `9`, como no DATASUS, e
  **`raca_cor`** traz `"Ignorado"` e campos em branco.

Como sabemos a verdade, dá para conferir cada resultado abaixo contra ela.

In [1]:
Random.seed!(20261011)
n = 1500

regiao      = rand(["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"], n)
data        = Date(2019, 1, 1) .+ Day.(rand(0:2_190, n))
idade       = rand(18:90, n)
escolaridade = [rand() < 0.12 ? 9 : rand(1:5) for _ in 1:n]     # 9 = ignorado (padrão DATASUS)
raca_cor    = [rand() < 0.08 ? "Ignorado" : rand() < 0.03 ? "" :
               rand(["Branca", "Preta", "Parda", "Amarela", "Indígena"]) for _ in 1:n]

renda     = Vector{Union{Missing,Int}}(round.(Int, 1200 .+ 90 .* (idade .- 18) .+ 700 .* randn(n)))
glicemia  = Vector{Union{Missing,Float64}}(round.(95 .+ 15 .* randn(n); digits = 1))
peso      = Vector{Union{Missing,Float64}}(round.(70 .+ 13 .* randn(n); digits = 1))

for i in 1:n
    # recusa em declarar renda: maior no Sudeste e entre os mais velhos
    rand() < 0.08 + 0.27 * (regiao[i] == "Sudeste") + 0.25 * (idade[i] > 65) && (renda[i] = missing)

    # consulta não realizada leva a glicemia e, quase sempre, o peso — mais em 2020–2021
    if rand() < 0.20 + 0.40 * (Date(2020, 3, 1) <= data[i] < Date(2022, 1, 1))
        glicemia[i] = missing
        rand() < 0.85 && (peso[i] = missing)
    end
end

df = DataFrame(; id = 1:n, regiao, data, idade, escolaridade, raca_cor, renda, glicemia, peso)
first(df, 6)

Row,id,regiao,data,idade,escolaridade,raca_cor,renda,glicemia,peso
,Int64,String,Date,Int64,Int64,String,Int64?,Float64?,Float64?
1,1,Sul,2019-09-04,80,1,Ignorado,6654,119.4,61.0
2,2,Sul,2022-04-20,41,3,Indígena,missing,80.9,57.7
3,3,Nordeste,2020-07-29,80,1,Branca,missing,missing,missing
4,4,Norte,2022-04-24,57,3,Parda,4916,102.7,79.7
5,5,Norte,2019-06-25,31,4,Branca,2813,97.8,72.2
6,6,Sudeste,2020-09-22,23,1,Amarela,1835,missing,107.9


## 1. Ausência disfarçada de valor — `missingsentinels`

Microdados administrativos raramente usam `missing`: a ausência vem como um código. Antes de qualquer
análise é preciso achar esses códigos, e `missingsentinels` propõe candidatos, cada um com a
evidência.

Um texto é proposto quando, sem espaços e em minúsculas, está vazio ou numa lista de termos de
ausência (`"na"`, `"null"`, `"ignorado"`, `"sem informação"`…). Um código numérico $c$ (`9`, `99`, …,
`-1`, `-9`, …) só é proposto quando os dados o sustentam. Sejam $x_{\max}$ e $x_{\min}$ os extremos
dos demais valores da coluna, e $g = 2$ se eles forem todos inteiros ($g = 1$ caso contrário). O
código é proposto se estiver isolado, depois de um vão:

$$
c > 0 \ \text{e}\ c - x_{\max} \ge g
\qquad\text{ou}\qquad
c < 0 \ \text{e}\ x_{\min} - c \ge g,
$$

ou se for muito mais frequente que os valores vizinhos:

$$
\frac{\#\{x = c\}}{\max\!\Bigl(1,\ \tfrac{1}{|K|}\sum_{k \in K} \#\{x = c + k\}\Bigr)} \ge 5,
\qquad K = \{\pm 1, \dots, \pm 5\}.
$$

Assim, `9` numa escala de 0 a 9, ou `99` numa coluna de idade que chega a 105, não é proposto.

In [2]:
found = missingsentinels(df)
DataFrame(found)

Row,column,value,n,pct,reason
,Symbol,Any,Int64,Float64,String
1,escolaridade,9,190,12.6667,above every other value (max 5)
2,raca_cor,Ignorado,110,7.33333,text that reads as absent
3,raca_cor,,48,3.2,blank text


Os três códigos plantados, e nada mais. São candidatos para revisão, não veredictos: fique com os que
são códigos e `sentinelisna` os transforma no `isna` por coluna que todas as funções do pacote
aceitam. A partir daqui, `9`, `"Ignorado"` e `""` contam como ausentes:

In [3]:
isna = sentinelisna(found)
missingsummary(df; isna, color = :never)

 column        type       missing        %  distribution
 glicemia      Float64        493   32.87%  ▃▃▃▃▃▃▃▄▃▃▃▃▃▃▄▃▃▃▃▃
 peso          Float64        416   27.73%  ▃▃▃▃▃▂▂▃▃▃▂▃▃▃▃▃▃▃▃▃
 renda         Int64          348   23.20%  ▂▃▂▃▂▂▂▃▃▃▃▂▃▃▂▂▂▂▂▂
 escolaridade  Int64          190   12.67%  ▂▁▂▂▂▂▂▁▂▂▁▁▂▁▁▁▂▂▁▁
 raca_cor      String         158   10.53%  ▁▂▁▂▂▁▁▂▁▂▁▁▂▁▁▁▁▁▁▁
 id            Int64            0    0.00%                      
 regiao        String           0    0.00%                      
 data          Date             0    0.00%                      
 idade         Int64            0    0.00%                      
 1605 missing of 13500 cells (11.89%) across 9 columns ┊ bins of 75 rows


## 2. A ausência é completamente aleatória? — `missingmcar`

Sob **MCAR** (*missing completely at random*, Rubin, 1976), o fato de um valor faltar não tem relação
com nenhum valor, observado ou não. O teste de Little (1988) agrupa as linhas por padrão de ausência
$s$ e compara a média observada de cada padrão, $\bar{\mathbf{y}}_s$, com a média estimada da tabela
inteira:

$$
d^2 = \sum_{s=1}^{S} n_s\,\bigl(\bar{\mathbf{y}}_s - \hat{\boldsymbol\mu}_{O_s}\bigr)^{\top}
      \hat{\boldsymbol\Sigma}_{O_sO_s}^{-1}\,\bigl(\bar{\mathbf{y}}_s - \hat{\boldsymbol\mu}_{O_s}\bigr),
\qquad d^2 \overset{H_0}{\sim} \chi^2_{\mathrm{gl}},\quad \mathrm{gl} = \sum_s p_s - p,
$$

em que $n_s$ é o número de linhas do padrão, $O_s$ as suas $p_s$ colunas observadas e $p$ o número de
colunas testadas. A média $\hat{\boldsymbol\mu}$ e a covariância $\hat{\boldsymbol\Sigma}$ são as
estimativas de máxima verossimilhança sob normal multivariada, obtidas pelo algoritmo EM (Dempster,
Laird e Rubin, 1977): no passo E, cada valor faltante é substituído pela sua regressão nos
observados,

$$
\hat{\mathbf{y}}_{M} = \boldsymbol\mu_M + \boldsymbol\Sigma_{MO}\boldsymbol\Sigma_{OO}^{-1}
                         \bigl(\mathbf{y}_O - \boldsymbol\mu_O\bigr),
$$

e no passo M a média e a covariância são recalculadas, até convergir. Um p-valor pequeno é evidência
*contra* MCAR.

In [4]:
r = missingmcar(df; columns = [:idade, :renda, :glicemia, :peso])
(estatistica = r.statistic, gl = r.df, p = r.pvalue, padroes = r.npatterns)

(estatistica = 105.81694611841141, gl = 11, p = 1.2501285228809797e-17, padroes = 6)

MCAR rejeitado com folga: a recusa da renda depende da idade, então as linhas sem renda são mais
velhas, e o teste vê isso nas médias por padrão.

Agora sem a renda, só a consulta:

In [5]:
r = missingmcar(df; columns = [:idade, :glicemia, :peso])
(estatistica = r.statistic, gl = r.df, p = r.pvalue, padroes = r.npatterns)

(estatistica = 3.361247812146434, gl = 3, p = 0.3392086582425562, padroes = 3)

Nenhuma evidência contra MCAR — e no entanto glicemia e peso **não** faltam ao acaso: faltam na
pandemia. O teste não tem como ver isso, porque nenhuma das colunas que recebeu depende da data.

Esse é o limite geral do teste: um p-valor pequeno é evidência contra MCAR; um p-valor grande só diz
que as colunas testadas não mostram rastro do mecanismo. Ele nunca prova MCAR. A seção seguinte acha
a data.

As estimativas do EM também vêm no resultado. Quando MCAR falha, a média dos casos observados é
viesada, e a do EM corrige pelo que as outras colunas dizem sobre os que faltam:

In [6]:
r = missingmcar(df; columns = [:idade, :renda, :glicemia, :peso])
media_obs(c) = sum(skipmissing(df[!, c])) / count(!ismissing, df[!, c])
DataFrame(coluna = r.columns,
          media_em = round.(r.mu; digits = 1),
          media_observada = [round(media_obs(c); digits = 1) for c in r.columns])

Row,coluna,media_em,media_observada
,Symbol,Float64,Float64
1,idade,54.2,54.2
2,renda,4509.9,4239.0
3,glicemia,94.8,94.8
4,peso,70.2,70.2


A renda observada é menor que a estimada pelo EM: quem recusou eram os mais velhos, e aqui os mais
velhos ganham mais. Usar só os casos completos subestimaria a renda média.

## 3. Do que a ausência depende? — `missingpredictors`

`missingmcar` diz *se* a ausência se relaciona com os dados; `missingpredictors` diz *com o quê*.
Para uma coluna-alvo, as linhas se dividem em dois grupos — 1, onde ela falta, e 2, onde está
presente — e cada outra coluna é comparada entre eles.

Para **números e datas**, o teste $t$ de Welch (1947), que não supõe variâncias iguais, com graus de
liberdade de Welch–Satterthwaite:

$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}},
\qquad
\nu = \frac{\bigl(s_1^2/n_1 + s_2^2/n_2\bigr)^2}
           {\dfrac{(s_1^2/n_1)^2}{n_1 - 1} + \dfrac{(s_2^2/n_2)^2}{n_2 - 1}},
$$

e o tamanho de efeito é a diferença padronizada de médias (Cohen, 1988), em que $0{,}2$ é pequeno,
$0{,}5$ médio e $0{,}8$ grande:

$$
d = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{(s_1^2 + s_2^2)/2}}.
$$

Para **categorias**, o $\chi^2$ de independência de Pearson (1900) numa tabela $2 \times k$, com o
$V$ de Cramér (1946) como efeito:

$$
\chi^2 = \sum_{g=1}^{2}\sum_{\ell=1}^{k} \frac{(O_{g\ell} - E_{g\ell})^2}{E_{g\ell}},
\qquad E_{g\ell} = \frac{n_\ell N_g}{N},
\qquad V = \sqrt{\frac{\chi^2}{N}}.
$$

In [7]:
preditores(col) = select(DataFrame(missingpredictors(df, col; isna)),
                         :column, :test, :pvalue, :effect, :detail)
preditores(:renda)

Row,column,test,pvalue,effect,detail
,Symbol,Symbol,Float64,Float64,String
1,regiao,chisq,2.34976e-24,0.27927,Sudeste: 37.9% of rows where missing vs 12.4% where present
2,idade,welch_t,4.2682e-23,0.629208,mean 63.99 where missing vs 51.29 where present
3,escolaridade,welch_t,0.00103425,0.213809,mean 3.228 where missing vs 2.927 where present
4,data,welch_t,0.179314,0.0821352,mean 2022-01-24 where missing vs 2021-12-02 where present
5,id,welch_t,0.324065,-0.0596768,mean 730.9 where missing vs 756.4 where present
6,raca_cor,chisq,0.594721,0.0496043,Parda: 21.8% of rows where missing vs 18.1% where present
7,peso,welch_t,0.679635,0.0290296,mean 70.48 where missing vs 70.1 where present
8,glicemia,welch_t,0.900458,-0.0091803,mean 94.65 where missing vs 94.79 where present


Os dois mecanismos da renda vêm no topo: a recusa está sobrerrepresentada no Sudeste, e quem
recusa é mais velho em média, com efeito médio ($d \approx 0{,}63$).

Logo abaixo aparece a **escolaridade**, com $p \approx 0{,}001$ — e nada na construção dos dados liga
escolaridade à recusa. É acaso. Com $m$ testes ao nível $\alpha$, o número esperado de falsos
positivos entre as colunas sem relação chega a $m\alpha$, e a correção de Bonferroni (Dunn, 1961)
exige

$$
p < \frac{\alpha}{m} = \frac{0{,}05}{8} \approx 0{,}006
$$

para manter a chance de *qualquer* falso positivo abaixo de $\alpha$. Aqui nem ela basta: a
escolaridade passa. Por isso o efeito importa tanto quanto o p-valor — $d \approx 0{,}21$ é pequeno,
um terço do da idade —, e num arquivo real, onde ninguém sabe a verdade, a última palavra é do
conhecimento do domínio e da replicação.

Agora a glicemia, que o teste de Little não conseguia explicar:

In [8]:
preditores(:glicemia)

Row,column,test,pvalue,effect,detail
,Symbol,Symbol,Float64,Float64,String
1,data,welch_t,7.2491e-10,-0.329088,mean 2021-08-02 where missing vs 2022-02-18 where present
2,regiao,chisq,0.0448478,0.0806244,Centro-Oeste: 25.6% of rows where missing vs 19.8% where present
3,peso,welch_t,0.38053,0.103004,mean 71.45 where missing vs 70.09 where present
4,renda,welch_t,0.489269,0.0429587,mean 4296 where missing vs 4210 where present
5,raca_cor,chisq,0.503701,0.0536936,Branca: 18.5% of rows where missing vs 16.7% where present
6,escolaridade,welch_t,0.911727,0.00653119,mean 3.005 where missing vs 2.995 where present
7,id,welch_t,0.945188,-0.00377099,mean 749.4 where missing vs 751 where present
8,idade,welch_t,0.957764,-0.00289657,mean 54.2 where missing vs 54.26 where present


A **data** vem em primeiro, com folga: a glicemia falta em registros de meados de 2021, no meio da
pandemia, contra o início de 2022 onde está presente. Comparando coluna a coluna, a própria data
entrega o mecanismo que escapou ao teste MCAR — e qualquer imputação da glicemia precisa
condicionar nela.

A região aparece em segundo, com $p \approx 0{,}045$ e $V \approx 0{,}08$: passa de raspão nos 5%, não
passaria por Bonferroni e tem efeito mínimo. Outra vez, acaso.

## 4. Padrões ordenados por estrutura — `missingpatterns(...; order = :cluster)`

`missingpatterns` lista cada combinação distinta de colunas presentes e ausentes, como o
`md.pattern()` do pacote `mice` do R. Com `order = :cluster`, as colunas que faltam juntas ficam lado
a lado. A associação entre as máscaras de ausência de duas colunas $a$ e $b$ é o coeficiente $\phi$
(Yule, 1912):

$$
\phi_{ab} = \frac{n_{ab}\,(n - n_a - n_b + n_{ab}) - (n_a - n_{ab})(n_b - n_{ab})}
                  {\sqrt{n_a(n - n_a)\,n_b(n - n_b)}},
$$

em que $n_a$ e $n_b$ contam as linhas sem $a$ e sem $b$, e $n_{ab}$ as linhas sem as duas. A ordem
começa pela coluna com mais ausências e acrescenta, a cada passo, a coluna de maior $\phi$ com a
última colocada.

In [9]:
missingpatterns(df; isna, order = :cluster, max_patterns = 10, color = :never)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃  glic…  ┃  peso   ┃  esco…  ┃  raca…  ┃  rend…  ┃   id    ┃  regi…  ┃  data   ┃  idad…  ┃    n    ┃    %    ┃  freq   ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   590   ┃  39.3%  ┃ ███████ ┃
┃  █████  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   256   ┃  17.1%  ┃ ███     ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   204   ┃  13.6%  ┃ ██      ┃
┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   83    ┃  5.5%   ┃ █       ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   76    ┃  5.1%   ┃ █       ┃
┃  █████  ┃  █████  ┃  ░

`glicemia` e `peso` ficam juntas, e os padrões em que as duas faltam se leem como um bloco só — a
consulta que não aconteceu.

## 5. Agrupando sem repetir o grupo — `by` e `exclude_by`

Com `by`, as linhas do mapa de calor viram grupos de uma coluna em vez de blocos de posições. Num
bloco que junta os grupos $G$ e as colunas $J$, a fração ausente pondera cada grupo pelo seu tamanho
$n_g$:

$$
\pi_{GJ} = \frac{\sum_{g \in G}\sum_{i \in g}\sum_{j \in J} m_{ij}}{|J|\sum_{g \in G} n_g},
$$

com $m_{ij} = 1$ quando o valor está ausente. Na 0.7 a própria coluna do grupo sai do mapa (ela só
repetiria o agrupamento); `exclude_by = false` a mantém.

In [10]:
plotmissing(df; by = :regiao, isna, layout = :classic, color = :never)

┏━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃            ┃    0%   ┃    0%   ┃    0%   ┃   13%   ┃   11%   ┃   23%   ┃   33%   ┃   28%   ┃
┣━━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃    row     ┃   id    ┃  data   ┃  idad…  ┃  esco…  ┃  raca…  ┃  rend…  ┃  glic…  ┃  peso   ┃
┣━━━━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃Centro-Oeste┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┃  Nordeste  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃
┃   Norte    ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃
┃  Sudeste   ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃
┃    Sul     ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃
┗━━━━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━

O Sudeste concentra a recusa de renda. Por ano, no HTML — que agora rotula as linhas da grade —,
aparece o buraco da pandemia na glicemia e no peso:

In [11]:
missingreport(df; by = :data, period = :year, isna, order = :cluster)

┏━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃     ┃   33%   ┃   28%   ┃   13%   ┃   11%   ┃   23%   ┃    0%   ┃    0%   ┃    0%   ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃ row ┃  glic…  ┃  peso   ┃  esco…  ┃  raca…  ┃  rend…  ┃   id    ┃  regi…  ┃  idad…  ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃2019 ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2020 ┃  █████  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2021 ┃  █████  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2022 ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2023 ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2024 ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┗━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┛

MissingPatterns.Analysis: 1500 × 8 table
 Grouping:    by data (year) → 6×8 cells
 Missing (count):            1605               ┊ Missing (%):          13.38%
 Present (count):           10395               ┊ Present (%):          86.62%
 Progress Bar:     [██████████████████████████████████████████████████████████]

## 6. Comparando duas versões do arquivo — `plotmissingdiff(...; on = :id)`

Bases públicas são republicadas: registros entram, saem, são corrigidos, e a ordem muda.
`plotmissingdiff` compara duas versões; com `on`, as linhas são pareadas pela chave em vez da
posição. Com as máscaras $m^{\text{antes}}$ e $m^{\text{depois}}$ das linhas pareadas, o resumo conta
exatamente

$$
\text{resolvidos} = \sum_{ij} m^{\text{antes}}_{ij}\bigl(1 - m^{\text{depois}}_{ij}\bigr),
\qquad
\text{introduzidos} = \sum_{ij} \bigl(1 - m^{\text{antes}}_{ij}\bigr) m^{\text{depois}}_{ij}.
$$

A segunda versão abaixo vem embaralhada, perde 60 notificações, ganha 40 novas, e passou por uma
revisão: 150 glicemias recuperadas e 20 pesos perdidos.

In [12]:
Random.seed!(7)
versao2 = df[shuffle(61:n), :]                    # reordenada, notificações 1–60 removidas

recuperadas = findall(ismissing, versao2.glicemia)[1:150]
versao2.glicemia[recuperadas] .= round.(95 .+ 15 .* randn(150); digits = 1)
perdidos = findall(!ismissing, versao2.peso)[1:20]
versao2.peso[perdidos] .= missing

novas = DataFrame(id = n+1:n+40, regiao = "Sul", data = Date(2025, 1, 1), idade = 40,
                  escolaridade = 3, raca_cor = "Parda", renda = 3500, glicemia = 92.0, peso = 72.0)
versao2 = vcat(versao2, novas)
size(df), size(versao2)

((1500, 9), (1480, 9))

In [13]:
plotmissingdiff(df, versao2; on = :id, color = :never)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ re… +0% ┃ da… +0% ┃ id… +0% ┃ es… +0% ┃ ra… +0% ┃ re… +0% ┃ g… -10% ┃ pe… +1% ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  +++++  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃
┃  ·····  ┃  ···

O resumo conta exatamente as edições feitas: 150 resolvidos e 20 introduzidos, nas 1.440
notificações que as duas versões compartilham, com as 60 removidas e as 40 novas informadas à parte.
Comparadas por posição, as mesmas tabelas alinhariam pessoas diferentes e mostrariam ruído.

## Referências

- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. <https://doi.org/10.1093/biomet/63.3.581>
- Little, R. J. A. (1988). A test of missing completely at random for multivariate data with missing values. *Journal of the American Statistical Association*, 83(404), 1198–1202. <https://doi.org/10.1080/01621459.1988.10478722>
- Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society: Series B*, 39(1), 1–22. <https://doi.org/10.1111/j.2517-6161.1977.tb01600.x>
- Little, R. J. A., & Rubin, D. B. (2019). *Statistical Analysis with Missing Data* (3rd ed.). Wiley. <https://doi.org/10.1002/9781119482260>
- Welch, B. L. (1947). The generalization of 'Student's' problem when several different population variances are involved. *Biometrika*, 34(1–2), 28–35. <https://doi.org/10.1093/biomet/34.1-2.28>
- Pearson, K. (1900). On the criterion that a given system of deviations from the probable in the case of a correlated system of variables is such that it can be reasonably supposed to have arisen from random sampling. *Philosophical Magazine*, 50(302), 157–175. <https://doi.org/10.1080/14786440009463897>
- Cramér, H. (1946). *Mathematical Methods of Statistics*. Princeton University Press. <https://doi.org/10.1515/9781400883868>
- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.). Lawrence Erlbaum; reimpresso em 2013, Routledge. <https://doi.org/10.4324/9780203771587>
- Dunn, O. J. (1961). Multiple comparisons among means. *Journal of the American Statistical Association*, 56(293), 52–64. <https://doi.org/10.1080/01621459.1961.10482090>
- Yule, G. U. (1912). On the methods of measuring association between two attributes. *Journal of the Royal Statistical Society*, 75(6), 579–652. <https://doi.org/10.2307/2340126>
- van Buuren, S., & Groothuis-Oudshoorn, K. (2011). mice: Multivariate imputation by chained equations in R. *Journal of Statistical Software*, 45(3). <https://doi.org/10.18637/jss.v045.i03>

As regras de `missingsentinels` são uma heurística do próprio pacote, não um teste publicado.

## Para continuar

- [Documentação]({DOCS}) — todas as funções e argumentos; a página [Methods]({DOCS}/methods/) traz
  a fórmula de cada número
- [`whats-new-0.7.ipynb`]({NB}whats-new-0.7.ipynb) — as mesmas novidades, em inglês
- [`getting-started.ipynb`]({NB}getting-started.ipynb) — o tour completo do pacote